# Лабораторная работа №1: Анализ данных в реляционной СУБД SQLite
**Дисциплина**: Основы искусственного интеллекта (ОИИ 26/27)  
**Учебное заведение**: ГОУ «ПГУ им. Т.Г. Шевченко», Физико-технический институт  
**Кафедра**: Информационных технологий  
**Студент**: Гандапас Даниил Владимирович  
**Группа**: ФТ24ДР62ПИ1  
**Вариант**: №1  
**База данных**: Northwind_large.sqlite  

---
### Содержание работы (Вариант №1):
1. **Вопрос 1**: Какой клиент сделал наибольшее количество заказов?
2. **Вопрос 2**: Сколько заказов было сделано в 2012 году?
3. **Вопрос 3**: Какие продукты ни разу не были заказаны?
4. **Вопрос 4**: Какой поставщик поставил наибольшее количество товаров в штуках?
5. **Вопрос 5**: Сколько сотрудников работают в каждом регионе?


## 0. Инициализация окружения и защита от повреждения БД
*Примечание*: Если блокнот запускается в Google Colab, ячейка автоматически скачивает оригинальную неповрежденную базу данных (32.2 МБ) напрямую на сервер и проверяет её целостность через `PRAGMA integrity_check`.


In [ ]:
import os
import sqlite3
import urllib.request
import pandas as pd
import matplotlib.pyplot as plt

DB_FILE = 'Northwind_large.sqlite'
EXPECTED_SIZE = 32235520 # Ровно 32.2 МБ

# 1. Если файл лежит локально в папке проекта
if not os.path.exists(DB_FILE) and os.path.exists(os.path.join('labs', 'oii_lab1', DB_FILE)):
    DB_FILE = os.path.join('labs', 'oii_lab1', DB_FILE)

# 2. Авто-скачивание при запуске в Google Colab или при битом файле (< 32 МБ)
needs_download = False
if not os.path.exists(DB_FILE):
    needs_download = True
    print("Файл базы данных не найден. Скачиваем оригинальную базу...")
elif os.path.getsize(DB_FILE) < EXPECTED_SIZE:
    needs_download = True
    print(f"ВНИМАНИЕ: Файл загружен не полностью ({os.path.getsize(DB_FILE)} из {EXPECTED_SIZE} байт). Скачиваем заново...")

if needs_download:
    url = "https://raw.githubusercontent.com/Asm-o-Dan/eli5-visual-hub/main/downloads/Northwind_large.sqlite"
    urllib.request.urlretrieve(url, 'Northwind_large.sqlite')
    DB_FILE = 'Northwind_large.sqlite'
    print(f"База успешно загружена: {os.path.getsize(DB_FILE)} байт.")

# 3. Подключение и проверка целостности
conn = sqlite3.connect(DB_FILE)
check_cursor = conn.cursor()
check_cursor.execute("PRAGMA integrity_check;")
status = check_cursor.fetchone()[0]
print(f"Подключение к БД успешно! Статус целостности: {status}")

# Список таблиц в базе
tables_df = pd.read_sql_query("""
    SELECT name AS TableName 
    FROM sqlite_master 
    WHERE type='table' AND name NOT LIKE 'sqlite_%'
    ORDER BY name;
""", conn)
display(tables_df)


Подключение к БД успешно! Статус целостности: ok


TableName
Category
Customer
CustomerCustomerDemo
CustomerDemographic
Employee
EmployeeTerritory
Order
OrderDetail
Product
Region


---
## Вопрос 1: Какой клиент сделал наибольшее количество заказов?
**Ход мысли:**  
Информация обо всех заказах хранится в таблице `"Order"` (название экранируем кавычками). В каждом заказе есть идентификатор клиента `CustomerId`. Названия компаний хранятся в таблице `Customer`.  
Соединяем `Order` и `Customer` по `CustomerId`, группируем по клиенту, считаем число заказов `COUNT(o.Id)` и сортируем по убыванию.


In [ ]:
query_q1 = """
SELECT 
    o.CustomerId,
    c.CompanyName,
    c.ContactName,
    c.City,
    c.Country,
    COUNT(o.Id) AS OrderCount
FROM "Order" o
LEFT JOIN Customer c ON o.CustomerId = c.Id
GROUP BY o.CustomerId
ORDER BY OrderCount DESC
LIMIT 10;
"""

df_q1 = pd.read_sql_query(query_q1, conn)
display(df_q1)

top_cust = df_q1.iloc[0]
print(f"Лидер по заказам: {top_cust['CompanyName']} ({top_cust['CustomerId']}) — {top_cust['OrderCount']} заказов.")


CustomerId,CompanyName,ContactName,City,Country,OrderCount
SAVEA,Save-a-lot Markets,Jose Pavarotti,Boise,USA,218
LAMAI,La maison d'Asie,Annette Roulet,Toulouse,France,217
QUICK,QUICK-Stop,Horst Kloss,Cunewalde,Germany,210
VAFFE,Vaffeljernet,Palle Ibsen,Århus,Denmark,207
QUEEN,Queen Cozinha,Lúcia Carvalho,Sao Paulo,Brazil,206
BERGS,Berglunds snabbköp,Christina Berglund,Luleå,Sweden,206
OLDWO,Old World Delicatessen,Rene Phillips,Anchorage,USA,205
MEREP,Mère Paillarde,Jean Fresnière,Montréal,Canada,205
LEHMS,Lehmanns Marktstand,Renate Messner,Frankfurt a.M.,Germany,204
ERNSH,Ernst Handel,Roland Mendel,Graz,Austria,204


Лидер по заказам: Save-a-lot Markets (SAVEA) — 218 заказов.


**Вывод к вопросу 1**:  
Наибольшее количество заказов сделал клиент **Save-a-lot Markets** (`CustomerId = 'SAVEA'`, город Boise, США) — **218 заказов**.  
*2-е место*: La maison d'Asie (`LAMAI`) — 217 заказов.  
*3-е место*: QUICK-Stop (`QUICK`) — 210 заказов.


---
## Вопрос 2: Сколько заказов было сделано в 2012 году?
**Ход мысли:**  
В таблице `"Order"` дата заказа лежит в столбце `OrderDate` в формате `YYYY-MM-DD`.  
Для точного подсчета заказов за 2012 год можно использовать фильтрацию по диапазону дат либо встроенную функцию SQLite `strftime('%Y', OrderDate)`.


In [ ]:
query_q2 = """
SELECT 
    strftime('%Y', OrderDate) AS Year,
    COUNT(Id) AS OrdersCount
FROM "Order"
WHERE strftime('%Y', OrderDate) = '2012'
GROUP BY Year;
"""

df_q2 = pd.read_sql_query(query_q2, conn)
display(df_q2)

orders_2012 = df_q2['OrdersCount'].iloc[0]
print(f"В 2012 году было сделано заказов: {orders_2012}")


Year,OrdersCount
2012,2323


В 2012 году было сделано заказов: 2323


**Вывод к вопросу 2**:  
В 2012 году было сделано ровно **2 323 заказа**.


---
## Вопрос 3: Какие продукты ни разу не были заказаны?
**Ход мысли:**  
Все существующие товары зафиксированы в таблице `Product`. Все когда-либо оформленные покупки товаров фиксируются в таблице чеков `OrderDetail` (поле `ProductId`).  
Чтобы найти товары, которых нет в чеках, используем **`LEFT JOIN`** таблицы `Product` с `OrderDetail` и фильтруем строки, где `OrderDetail.Id IS NULL` (либо через подзапрос `WHERE Id NOT IN (SELECT DISTINCT ProductId FROM OrderDetail)`).


In [ ]:
query_q3 = """
SELECT 
    p.Id AS ProductId,
    p.ProductName,
    p.UnitPrice
FROM Product p
LEFT JOIN OrderDetail od ON p.Id = od.ProductId
WHERE od.Id IS NULL;
"""

df_q3 = pd.read_sql_query(query_q3, conn)
display(df_q3)

if len(df_q3) == 0:
    print("В базе данных НЕТ незаказанных продуктов: все 77 продуктов были заказаны хотя бы один раз!")
else:
    print(f"Найдено незаказанных продуктов: {len(df_q3)}")


ProductId,ProductName,UnitPrice


В базе данных НЕТ незаказанных продуктов: все 77 продуктов были заказаны хотя бы один раз!


**Вывод к вопросу 3**:  
В базе данных **нет ни одного незаказанного продукта** (результирующая выборка пуста, 0 строк).  
Все **77 товаров** из каталога `Product` были заказаны покупателями как минимум один раз (100% реализация ассортимента).


---
## Вопрос 4: Какой поставщик поставил наибольшее количество товаров в штуках?
**Ход мысли:**  
Поставщики хранятся в таблице `Supplier`. Каждый товар в таблице `Product` имеет внешний ключ `SupplierId`. Фактическое количество проданных единиц в штуках хранится в таблице `OrderDetail` в столбце `Quantity`.  
Связываем три таблицы: `Supplier → Product → OrderDetail`, группируем по поставщику и суммируем количество проданных единиц: `SUM(od.Quantity)`.


In [ ]:
query_q4 = """
SELECT 
    s.Id AS SupplierId,
    s.CompanyName,
    s.Country,
    SUM(od.Quantity) AS TotalUnitsSupplied
FROM Supplier s
JOIN Product p ON s.Id = p.SupplierId
JOIN OrderDetail od ON p.Id = od.ProductId
GROUP BY s.Id, s.CompanyName, s.Country
ORDER BY TotalUnitsSupplied DESC
LIMIT 10;
"""

df_q4 = pd.read_sql_query(query_q4, conn)
display(df_q4)

top_sup = df_q4.iloc[0]
print(f"Поставщик-лидер: {top_sup['CompanyName']} — {top_sup['TotalUnitsSupplied']:,} штук.")


SupplierId,CompanyName,Country,TotalUnitsSupplied
7,"Pavlova, Ltd.",Australia,1033909
12,Plutzer Lebensmittelgroßmärkte AG,Germany,1029720
8,"Specialty Biscuits, Ltd.",UK,824312
2,New Orleans Cajun Delights,USA,821051
15,Norske Meierier,Norway,623347
1,Exotic Liquids,UK,622672
24,"G'day, Mate",Australia,620886
4,Tokyo Traders,Japan,619050
11,Heli Süßwaren GmbH & Co. KG,Germany,618710
16,Bigfoot Breweries,USA,618636


Поставщик-лидер: Pavlova, Ltd. — 1,033,909 штук.


**Вывод к вопросу 4**:  
Наибольшее количество товаров в штуках поставила компания **Pavlova, Ltd.** (`SupplierId = 7`, Австралия) — **1 033 909 штук**.  
*2-е место*: Plutzer Lebensmittelgroßmärkte AG (`SupplierId = 12`, Германия) — 1 029 720 штук.  
*3-е место*: Specialty Biscuits, Ltd. (`SupplierId = 8`, Великобритания) — 824 312 штук.


---
## Вопрос 5: Сколько сотрудников работают в каждом регионе?
**Ход мысли:**  
В реляционной схеме Northwind понятие «регион» представлено двумя способами:
1. **Прямое поле в таблице сотрудников**: `Employee.Region` (макро-регионы проживания/офиса).
2. **Связка через территории обслуживания**: `Region → Territory → EmployeeTerritory → Employee` (географические зоны продаж).  
Приведем оба среза данных для исчерпывающего ответа на защите.


In [ ]:
# Вариант А: Прямой срез по полю Employee.Region
query_q5_a = """
SELECT 
    COALESCE(Region, 'Не указан') AS RegionName,
    COUNT(Id) AS EmployeeCount
FROM Employee
GROUP BY Region;
"""
df_q5_a = pd.read_sql_query(query_q5_a, conn)
print("=== Срез 1: По полю Employee.Region ===")
display(df_q5_a)

# Вариант Б: Реляционная связка Region -> Territory -> EmployeeTerritory
query_q5_b = """
SELECT 
    r.Id AS RegionId,
    r.RegionDescription,
    COUNT(DISTINCT et.EmployeeId) AS EmployeeCount
FROM Region r
JOIN Territory t ON r.Id = t.RegionId
JOIN EmployeeTerritory et ON t.Id = et.TerritoryId
GROUP BY r.Id, r.RegionDescription
ORDER BY EmployeeCount DESC;
"""
df_q5_b = pd.read_sql_query(query_q5_b, conn)
print("\n=== Срез 2: По зонам ответственности (Region / Territory) ===")
display(df_q5_b)


=== Срез 1: По полю Employee.Region ===


RegionName,EmployeeCount
British Isles,4
North America,5



=== Срез 2: По зонам ответственности (Region / Territory) ===


RegionId,RegionDescription,EmployeeCount
1,Eastern,4
2,Western,2
3,Northern,2
4,Southern,1


**Вывод к вопросу 5**:  
* **По полю `Employee.Region`** (всего 9 сотрудников):
  * **North America**: **5 сотрудников**
  * **British Isles**: **4 сотрудника**
* **По зонам ответственности торговых территорий (`Region` $\to$ `Territory`)**:
  * **Eastern**: **4 сотрудника**
  * **Western**: **2 сотрудника**
  * **Northern**: **2 сотрудника**
  * **Southern**: **1 сотрудник**


---
## ИТОГОВЫЙ ПАСПОРТ РЕШЕНИЯ (ВАРИАНТ №1)

| № | Вопрос лабораторной работы | Точный ответ | Числовое значение / Детали |
|---|----------------------------|--------------|----------------------------|
| **1** | Какой клиент сделал наибольшее количество заказов? | **Save-a-lot Markets** (`SAVEA`) | **218 заказов** (США, Boise) |
| **2** | Сколько заказов было сделано в 2012 году? | **2 323 заказа** | Фильтр `strftime('%Y', OrderDate) = '2012'` |
| **3** | Какие продукты ни разу не были заказаны? | **0 продуктов** (таких нет) | Все 77 продуктов заказаны $\ge 1$ раза |
| **4** | Какой поставщик поставил наибольшее количество товаров в штуках? | **Pavlova, Ltd.** (Id=7) | **1 033 909 штук** |
| **5** | Сколько сотрудников работают в каждом регионе? | **North America: 5, British Isles: 4** | Eastern: 4, Western: 2, Northern: 2, Southern: 1 |
